# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anrajvarun9838/flyrank_internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


I will frame this as a **classification** task. The goal is to classify each page as declining or not declining using measurable page and search signals. Classification fits because the target is a binary outcome defined by the available trend label. The model can help prioritize pages for further human review and decision-support.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1 check

task_type = "Binary classification"
target_classes = ["declining", "not declining"]

print("ML task:", task_type)
print("Target classes:", target_classes)


ML task: Binary classification
Target classes: ['declining', 'not declining']


## 2. Target or proxy

The target is the page's **trend direction**. The starter dataset provides `trend_direction`, which is an observed dataset field describing whether the page is trending up, flat, or down.

For this ML task, I will define a binary proxy target:

- `1` = declining (`trend_direction == "down"`)
- `0` = not declining (up or flat)

This is a **defined proxy label**, not a future outcome. Therefore, the model should be evaluated as a prioritization and decision-support tool rather than as proof that a page will decline in the future.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Load the official FlyRank starter dataset

import os
import pandas as pd
import subprocess

REPO_DIR = "/content/flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
            REPO_DIR,
        ],
        check=True,
    )

csv_path = os.path.join(
    REPO_DIR,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(csv_path)

# Create binary proxy target
df["target_declining"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df["target_declining"].value_counts())

print("\nTarget rate:")
print(df["target_declining"].mean())

Dataset shape: (30000, 45)

Target distribution:
target_declining
1    16262
0    13738
Name: count, dtype: int64

Target rate:
0.5420666666666667


## 3. Success metric

My primary success metric will be **PR-AUC (Average Precision)**.

PR-AUC is useful because the practical task is to identify pages that belong to the declining class and prioritize them for review. It evaluates the precision-recall trade-off across classification thresholds rather than depending on one fixed threshold.

A higher PR-AUC means the model is better at putting true declining pages higher in its predicted list while limiting false positives.

The exact value that counts as "good" will be decided using a simple baseline and a held-out validation set rather than choosing an arbitrary number before seeing the data.

In [8]:
# Section 3 check

from sklearn.metrics import average_precision_score

positive_rate = df["target_declining"].mean()

print("Primary metric: PR-AUC / Average Precision")
print(f"Positive-class baseline rate: {positive_rate:.3f}")

print(
    "\nA useful model should be evaluated against this baseline "
    "on held-out data."
)

Primary metric: PR-AUC / Average Precision
Positive-class baseline rate: 0.542

A useful model should be evaluated against this baseline on held-out data.


## 4. The unit of analysis

The unit of analysis is **one content page**.

Therefore, one row in the dataframe represents one page-level observation with its measured search and content signals.

The target is attached to the same page-level observation. I will use page-level features to study which observed signals are associated with the declining proxy label.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4 check: show the real dataframe

print("One row = one content page")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

display(df.head())

print("\nData types:")
display(df.dtypes)


One row = one content page
Number of rows: 30000
Number of columns: 45


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,target_declining
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1



Data types:


,0
content_id,object
client_id,object
search_volume,float64
competition,float64
competition_level,object
cpc,float64
content_type,object
main_intent,object
word_count,float64
char_count,float64


## 5. Why ML beats a fixed rule here

A single fixed rule, such as "mark every page with fewer than X impressions as declining," would use only one signal and one arbitrary cutoff.

The dataset contains multiple measurable signals that can interact in different ways. A page may have high impressions but a worsening trend, while another page may have lower impressions but a different combination of signals.

ML is useful here because it can combine multiple signals and learn patterns from the labeled examples instead of relying on one manually chosen threshold.

The goal is not to claim that ML explains Google's ranking system. The goal is to test whether a measured model can provide better directional prioritization than a simple fixed-rule baseline.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 5 check

# Show the number of numeric features available for a multi-signal model.

numeric_features = df.select_dtypes(include="number").columns.tolist()

# Remove the target itself from the feature count
numeric_features = [
    col for col in numeric_features
    if col != "target_declining"
]

print("Number of numeric signals available:", len(numeric_features))
print("\nNumeric signals:")
print(numeric_features)

print(
    "\nReason for ML:",
    "multiple measurable signals can be combined instead of using one fixed threshold."
)

Number of numeric signals available: 30

Numeric signals:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'trend_pct']

Reason for ML: multiple measurable signals can be combined instead of using one fixed threshold.


## Self-check

Before you submit, confirm each line honestly:
- ☑️ Every section above is filled — markdown thinking AND the code that backs it
- ⬜ The notebook runs top to bottom with no errors
- ☑️ No client names, URLs, or private queries anywhere
- ☑️ My claims use careful words: observed, measured, directional, decision-support
- ⬜ Committed to my repo under `work/notebooks/`